# Segmentation of frequency interval where signal to be corrected is present

In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals
from Functions.time_frequency import spectrogram
import pywt
import cv2
from sklearn.decomposition import NMF
from specparam import SpectralModel # for FOOOF method

## 1. Generate synthetic signal

We generate a synthetic signal. In the EEG we want to mimic the presence of $\delta$ + $\alpha$ bands, as it would be for a gabaergic anesthesia and the addition of a $\beta$ band to mimic the __ketamine signature__.

To generate one band we use a __2D Ornstein-Ulhenbeck__ process that can create spindles like patterns with waning and waxing envelopes containing oscillations.

In [4]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 4]
factor_list = [1, 1, 2]

# --- Generate EEG data
t, y = get_mixed_OU_signals(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)
# mask_f = f_spectro >= 20
# f_spectro = f_spectro[mask_f]
# spectro = spectro[mask_f, :]

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.sum(spectro, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

## FOOOF method

Using the FOOOF method one can have an automatic estimate of the peak bump in the signal as well as an approximation of the baseline.

However it seems, basleine can be way higher than psd parts. --> maybe use different method to have baseline mostly bellow psd

In [ ]:
SMOOTH = True

f_psd = f_spectro
psd = np.mean(spectro, axis=1)
freq_range = [0.1, 45]
to_fit = psd

if SMOOTH: # smooth to have easier fit of the PSD
    psd_smoothed = sc.signal.savgol_filter(psd, 5, 1)
    to_fit = psd_smoothed

fm = SpectralModel(peak_width_limits=[1.0, 15.0], max_n_peaks=4, min_peak_height=0.25)
fm.fit(f_psd, to_fit, freq_range)
fm.report(f_psd, to_fit, freq_range)

# --- Extract model components (specparam 2.0 nested API) ---
freqs = fm.data.freqs                          # frequency values used in the fit
psd_log = fm.data.power_spectrum               # original PSD, log10 power
full_fit = fm.results.model.modeled_spectrum   # full model fit, log10 power
ap_fit = fm.results.model._ap_fit              # aperiodic component only, log10 power

# Peak params: columns are [CF, PW, BW] (BW = bandwidth, ~2*std, for gaussian mode)
peak_params = fm.get_params('periodic')
peak_params = np.atleast_2d(peak_params)

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(freqs, psd_log, color='black', lw=1.5, label='Original PSD')
ax.plot(freqs, full_fit, color='red', lw=2, label='Full model fit')
ax.plot(freqs, ap_fit, color='blue', lw=1.5, ls='--', label='Aperiodic fit')

colors = plt.cm.viridis(np.linspace(0, 1, max(len(peak_params), 1)))

for i, (cf, pw, bw) in enumerate(peak_params):
    bw = 5*bw/2
    left = cf - bw / 2
    right = cf + bw / 2

    ax.axvline(cf, color=colors[i], ls=':', lw=1)
    ax.axvspan(left, right, color=colors[i], alpha=0.15,
               label=f'Peak {i+1}: {cf:.2f} Hz [{left:.2f}-{right:.2f}]')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('log10(Power)')
ax.set_title('Spectral Parameterization')
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

                                                                                                  
                                      SPECTRUM MODEL RESULTS                                      
                                                                                                  
                       The model was fit with the 'spectral_fit' algorithm                        
               Model was fit to the 0-45 Hz frequency range with 0.50 Hz resolution               
                                                                                                  
                               Aperiodic Parameters ('fixed' mode)                                
                                        (offset, exponent)                                        
                                         -0.3419, 1.0728                                          
                                                                                                  
          

## Source separation with NMF

In [5]:
M = spectro.copy()

M = spectro ** 0.33

In [6]:
# --- 1. Prepare Positive Input Data for NMF ---
# Ensure the spectrogram is real and strictly non-negative (Magnitude spectrogram)
V = np.abs(M) 

# --- 2. Fit NMF ---
# We have 3 distinct components generated in your parameters (lbda_list, etc.)
n_components = 3

# We initialize with 'nndsvdar' which is excellent for spectrograms (handling sparse data)
nmf_model = NMF(n_components=n_components, init='nndsvdar', random_state=42, max_iter=1000)

# W (dictionary matrix) represents the spectral signatures of our sources (frequency profiles)
# H (activation matrix) represents the temporal evolution of these sources over time
W = nmf_model.fit_transform(V) # Shape: (frequencies, components)
H = nmf_model.components_      # Shape: (components, time_bins)

# --- 3. Reconstruct Individual Sources ---
# To see what each isolated source looks like in the spectrogram domain:
reconstructed_sources = []
for i in range(n_components):
    # Outer product of the i-th frequency profile and the i-th time activation
    source_i = W[:, [i]] @ H[[i], :]
    reconstructed_sources.append(source_i)

# --- 4. Visualization ---
fig, axes = plt.subplots(n_components + 2, 2, figsize=(12, 2 * n_components + 4), sharex='col', constrained_layout=True)

# Plot Original Spectrogram for reference
axes[0, 0].pcolormesh(t_spectro, f_spectro, np.log2(V + 1e-11), shading='nearest', cmap='jet')
axes[0, 0].set_title('Original Spectrogram')
axes[0, 1].axis('off') # Keep layout clean

# Plot NMF Components (W and H) and Reconstructed Sources
for i in range(n_components):
    # Frequency Profiles (W) - Left Column
    axes[i+1, 0].plot(f_spectro, W[:, i], color='darkorange', lw=2)
    axes[i+1, 0].set_title(f'Component {i+1} Frequency Profile (W)')
    axes[i+1, 0].set_ylabel('Amplitude')
    
    # Time Activations (H) - Right Column
    axes[i+1, 1].plot(t_spectro, H[i, :], color='dodgerblue', lw=2)
    axes[i+1, 1].set_title(f'Component {i+1} Time Activation (H)')
    axes[i+1, 1].set_ylabel('Activation')

# Plot the reconstructed full spectrogram (W * H) to see reconstruction quality
V_approx = W @ H
axes[-1, 0].pcolormesh(t_spectro, f_spectro, np.log2(V_approx + 1e-11), shading='nearest', cmap='jet')
axes[-1, 0].set_title('Reconstructed Spectrogram (W * H)')
axes[-1, 0].set_xlabel('Time (s)')
axes[-1, 1].axis('off')

plt.show()

C:\Users\holcman\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\decomposition\_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


## Methods to detect frequency intervals of ketamine signature.

### Mask with PSD

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage.filters import threshold_multiotsu

# --- 1. Compute Base Data & Log Transforms ---
# Compute 1D PSD (using median across time)
psd = np.median(spectro, axis=1)
log2_psd = np.log2(psd + 1e-11)

# 2D Spectrogram log transform
log2_spectro = np.log2(spectro + 1e-11)


# --- 2. Apply Multi-Otsu Method A: On 1D PSD ---
otsu_thresh_1d = threshold_multiotsu(log2_psd, classes=4)
linear_thresh_1d = 2 ** otsu_thresh_1d[1]
# Broadcast the 1D threshold across the time axis
mask_1d = np.full_like(spectro, linear_thresh_1d)
masked_spectro_by_psd = np.where(spectro >= mask_1d, spectro, 0.0)


# --- 3. Apply Multi-Otsu Method B: Directly on 2D Spectrogram ---
otsu_thresh_2d = threshold_multiotsu(log2_spectro, classes=4)
linear_thresh_2d = 2 ** otsu_thresh_2d[1]
masked_spectro_direct_2d = np.where(spectro >= linear_thresh_2d, spectro, 0.0)


# --- 4. Setup GridSpec Figure Layout ---
# 5 rows total. Column 0 contains time-domain/spectrogram plots. 
# Column 1 contains frequency-domain/projection plots.
fig = plt.figure(figsize=(14, 15))
gs = GridSpec(5, 2, figure=fig, height_ratios=[1.2, 2, 2, 2, 1.5], width_ratios=[3, 1])

# Create axes
ax_signal      = fig.add_subplot(gs[0, 0])
ax_empty_top   = fig.add_subplot(gs[0, 1]) 

ax_orig        = fig.add_subplot(gs[1, 0], sharex=ax_signal)
ax_empty_mid   = fig.add_subplot(gs[1, 1]) 

# Row 2: PSD-Masked Spectrogram + Continuous 1D PSD Plot (sharing Frequency/Y axis)
ax_mask_psd    = fig.add_subplot(gs[2, 0], sharex=ax_signal, sharey=ax_orig)
ax_psd_line    = fig.add_subplot(gs[2, 1], sharey=ax_orig)

# Row 3: 2D Masked Spectrogram
ax_mask_2d     = fig.add_subplot(gs[3, 0], sharex=ax_signal, sharey=ax_orig)
ax_empty_bot   = fig.add_subplot(gs[3, 1])

# Row 4: Diagnostic Histograms at the absolute bottom
ax_hist_1d     = fig.add_subplot(gs[4, 0])
ax_hist_2d     = fig.add_subplot(gs[4, 1])

# Clean up empty spaces
ax_empty_top.axis('off')
ax_empty_mid.axis('off')
ax_empty_bot.axis('off')


# --- 5. Plotting Time-Domain & Spectrograms (Col 0) ---

# Plot 1: Raw EEG Signal
ax_signal.plot(t, y, color='black', alpha=0.75, lw=1)
ax_signal.set_title('Simulated EEG Signal', fontsize=12, fontweight='bold')
ax_signal.set_ylabel('Amplitude')

# Plot 2: Original Spectrogram
ax_orig.pcolormesh(t_spectro, f_spectro, log2_spectro, shading='nearest', cmap='jet')
ax_orig.set_title('Original Spectrogram', fontsize=12, fontweight='bold')
ax_orig.set_ylabel('Frequency (Hz)')

# Plot 3: Method A - Masked via 1D PSD Threshold
ax_mask_psd.pcolormesh(t_spectro, f_spectro, np.log2(masked_spectro_by_psd + 1e-11), shading='nearest', cmap='jet')
ax_mask_psd.set_title(f'Method A: Masked via 1D PSD Threshold', fontsize=11, color='navy', fontweight='bold')
ax_mask_psd.set_ylabel('Frequency (Hz)')

# Plot 4: Method B - Masked Directly on 2D Pixels
ax_mask_2d.pcolormesh(t_spectro, f_spectro, np.log2(masked_spectro_direct_2d + 1e-11), shading='nearest', cmap='jet')
ax_mask_2d.set_title(f'Method B: Masked on 2D Spectrogram Directly', fontsize=11, color='purple', fontweight='bold')
ax_mask_2d.set_ylabel('Frequency (Hz)')
ax_mask_2d.set_xlabel('Time (s)')


# --- 6. Plotting Continuous 1D PSD with Boundaries (Row 2, Col 1) ---
ax_psd_line.plot(log2_psd, f_spectro, color='navy', lw=2, label='PSD (Log2)')
ax_psd_line.axvline(otsu_thresh_1d[0], color='orange', linestyle='--', lw=1.5, label='Thresh 1')
ax_psd_line.axvline(otsu_thresh_1d[1], color='red', linestyle='--', lw=1.5, label='Thresh 2 (Cut)')
ax_psd_line.set_title('1D PSD Profile', fontsize=11)
ax_psd_line.set_xlabel('Log2 Power')
ax_psd_line.legend(fontsize=8, loc='upper right')
ax_psd_line.grid(True, alpha=0.4)


# --- 7. Diagnostic Distributions (Row 4) ---

# Bottom Left: 1D PSD Histogram 
ax_hist_1d.hist(log2_psd, bins=30, color='navy', alpha=0.7, edgecolor='black')
ax_hist_1d.axvline(otsu_thresh_1d[0], color='orange', linestyle='--')
ax_hist_1d.axvline(otsu_thresh_1d[1], color='red', linestyle='--')
ax_hist_1d.set_title('Multi-Otsu Decision Graph on 1D PSD Values')
ax_hist_1d.set_xlabel('Log2 PSD Power')
ax_hist_1d.set_ylabel('Bin Count')
ax_hist_1d.grid(True, alpha=0.4)

# Bottom Right: 2D Pixel Histogram
ax_hist_2d.hist(log2_spectro.flatten(), bins=50, color='purple', alpha=0.7, edgecolor='black')
ax_hist_2d.axvline(otsu_thresh_2d[0], color='orange', linestyle='--')
ax_hist_2d.axvline(otsu_thresh_2d[1], color='red', linestyle='--')
ax_hist_2d.set_title('Multi-Otsu on 2D Pixels')
ax_hist_2d.set_xlabel('Log2 Pixel Value')
ax_hist_2d.grid(True, alpha=0.4)

# Clean rendering
plt.tight_layout()
plt.show()

In [69]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from skimage.filters import threshold_multiotsu

# --- 1. Prepare Positive Input Data for NMF ---
V = np.abs(M) 

# --- 2. Fit NMF ---
n_components = 10
nmf_model = NMF(n_components=n_components, init='nndsvdar', random_state=42, max_iter=1000)

W = nmf_model.fit_transform(V) 
H = nmf_model.components_      

M = W @ H


# --- 1. Compute Base Data & Log Transforms from Matrix M ---
# Compute 1D PSD (using median across time)
psd = np.median(M, axis=1)
log2_psd = np.log2(psd + 1e-11)

# 2D Spectrogram log transform
log2_M = np.log2(M + 1e-11)


# --- 2. Apply Multi-Otsu Method A: On 1D PSD ---
otsu_thresh_1d = threshold_multiotsu(log2_psd, classes=3)
linear_thresh_1d = 2 ** otsu_thresh_1d[1]
# Broadcast the 1D threshold across the time axis
mask_1d = np.full_like(M, linear_thresh_1d)
masked_M_by_psd = np.where(M >= mask_1d, M, 0.0)


# --- 3. Apply Multi-Otsu Method B: Directly on 2D Matrix M ---
otsu_thresh_2d = threshold_multiotsu(log2_M, classes=3)
linear_thresh_2d = 2 ** otsu_thresh_2d[1]
masked_M_direct_2d = np.where(M >= linear_thresh_2d, M, 0.0)


# --- 4. Setup GridSpec Figure Layout ---
# 5 rows total. Column 0 contains time-domain/spectrogram plots. 
# Column 1 contains frequency-domain/projection plots.
fig = plt.figure(figsize=(14, 15))
gs = GridSpec(5, 2, figure=fig, height_ratios=[1.2, 2, 2, 2, 1.5], width_ratios=[3, 1])

# Create axes
ax_signal      = fig.add_subplot(gs[0, 0])
ax_empty_top   = fig.add_subplot(gs[0, 1]) 

ax_orig        = fig.add_subplot(gs[1, 0], sharex=ax_signal)
ax_empty_mid   = fig.add_subplot(gs[1, 1]) 

# Row 2: PSD-Masked Spectrogram + Continuous 1D PSD Plot (sharing Frequency/Y axis)
ax_mask_psd    = fig.add_subplot(gs[2, 0], sharex=ax_signal, sharey=ax_orig)
ax_psd_line    = fig.add_subplot(gs[2, 1], sharey=ax_orig)

# Row 3: 2D Masked Spectrogram
ax_mask_2d     = fig.add_subplot(gs[3, 0], sharex=ax_signal, sharey=ax_orig)
ax_empty_bot   = fig.add_subplot(gs[3, 1])

# Row 4: Diagnostic Histograms at the absolute bottom
ax_hist_1d     = fig.add_subplot(gs[4, 0])
ax_hist_2d     = fig.add_subplot(gs[4, 1])

# Clean up empty spaces
ax_empty_top.axis('off')
ax_empty_mid.axis('off')
ax_empty_bot.axis('off')


# --- 5. Plotting Time-Domain & Spectrograms (Col 0) ---

# Plot 1: Raw EEG Signal
ax_signal.plot(t, y, color='black', alpha=0.75, lw=1)
ax_signal.set_title('Simulated EEG Signal', fontsize=12, fontweight='bold')
ax_signal.set_ylabel('Amplitude')

# Plot 2: Original Spectrogram (Matrix M)
ax_orig.pcolormesh(t_spectro, f_spectro, log2_M, shading='nearest', cmap='jet')
ax_orig.set_title('Original Spectrogram (M)', fontsize=12, fontweight='bold')
ax_orig.set_ylabel('Frequency (Hz)')

# Plot 3: Method A - Masked via 1D PSD Threshold
ax_mask_psd.pcolormesh(t_spectro, f_spectro, np.log2(masked_M_by_psd + 1e-11), shading='nearest', cmap='jet')
ax_mask_psd.set_title(f'Method A: Masked via 1D PSD Threshold', fontsize=11, color='navy', fontweight='bold')
ax_mask_psd.set_ylabel('Frequency (Hz)')

# Plot 4: Method B - Masked Directly on 2D Pixels
ax_mask_2d.pcolormesh(t_spectro, f_spectro, np.log2(masked_M_direct_2d + 1e-11), shading='nearest', cmap='jet')
ax_mask_2d.set_title(f'Method B: Masked on 2D Spectrogram Directly', fontsize=11, color='purple', fontweight='bold')
ax_mask_2d.set_ylabel('Frequency (Hz)')
ax_mask_2d.set_xlabel('Time (s)')


# --- 6. Plotting Continuous 1D PSD with Boundaries (Row 2, Col 1) ---
ax_psd_line.plot(log2_psd, f_spectro, color='navy', lw=2, label='PSD (Log2)')
ax_psd_line.axvline(otsu_thresh_1d[0], color='orange', linestyle='--', lw=1.5, label='Thresh 1')
ax_psd_line.axvline(otsu_thresh_1d[1], color='red', linestyle='--', lw=1.5, label='Thresh 2 (Cut)')
ax_psd_line.set_title('1D PSD Profile', fontsize=11)
ax_psd_line.set_xlabel('Log2 Power')
ax_psd_line.legend(fontsize=8, loc='upper right')
ax_psd_line.grid(True, alpha=0.4)


# --- 7. Diagnostic Distributions (Row 4) ---

# Bottom Left: 1D PSD Histogram 
ax_hist_1d.hist(log2_psd, bins=30, color='navy', alpha=0.7, edgecolor='black')
ax_hist_1d.axvline(otsu_thresh_1d[0], color='orange', linestyle='--')
ax_hist_1d.axvline(otsu_thresh_1d[1], color='red', linestyle='--')
ax_hist_1d.set_title('Multi-Otsu Decision Graph on 1D PSD Values')
ax_hist_1d.set_xlabel('Log2 PSD Power')
ax_hist_1d.set_ylabel('Bin Count')
ax_hist_1d.grid(True, alpha=0.4)

# Bottom Right: 2D Pixel Histogram
ax_hist_2d.hist(log2_M.flatten(), bins=50, color='purple', alpha=0.7, edgecolor='black')
ax_hist_2d.axvline(otsu_thresh_2d[0], color='orange', linestyle='--')
ax_hist_2d.axvline(otsu_thresh_2d[1], color='red', linestyle='--')
ax_hist_2d.set_title('Multi-Otsu on 2D Pixels (M)')
ax_hist_2d.set_xlabel('Log2 Pixel Value')
ax_hist_2d.grid(True, alpha=0.4)

# Clean rendering
plt.tight_layout()
plt.show()

### Low component NMF

In [84]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from sklearn.decomposition import NMF

# --- 1. Prepare Positive Input Data for NMF ---
V = np.abs(M) 

# --- 2. Fit NMF ---
n_components = 3
#nmf_model = NMF(n_components=n_components, init='nndsvdar', random_state=42, max_iter=1000) # Froebenius cost
nmf_model = NMF(n_components=n_components, beta_loss='kullback-leibler', solver='mu', init='nndsvdar', random_state=42, max_iter=1000)

W = nmf_model.fit_transform(V) 
H = nmf_model.components_      

V_approx = W @ H

# --- 4. Visualization with GridSpec (Added 1 extra row for distributions) ---
# We now have (n_components + 3) rows total
n_rows = n_components + 3
fig = plt.figure(figsize=(12, 2 * n_rows + 2))
gs = GridSpec(n_rows, 2, figure=fig, width_ratios=[3, 1])

# Create existing axes
ax_orig_spectro = fig.add_subplot(gs[0, 0])
ax_orig_psd = fig.add_subplot(gs[0, 1], sharey=ax_orig_spectro)

ax_comps = []
for i in range(n_components):
    ax_w = fig.add_subplot(gs[i+1, 0])                            
    ax_h = fig.add_subplot(gs[i+1, 1], sharex=ax_orig_spectro)    
    ax_comps.append((ax_w, ax_h))

ax_recon_spectro = fig.add_subplot(gs[-2, 0], sharex=ax_orig_spectro, sharey=ax_orig_spectro)
ax_recon_psd = fig.add_subplot(gs[-2, 1], sharey=ax_recon_spectro)

# --- NEW: Create Axes for Bins Distribution (Bottom Row) ---
ax_dist_orig = fig.add_subplot(gs[-1, 0])
ax_dist_recon = fig.add_subplot(gs[-1, 1])


# --- 5. Plotting Content ---

# 1. Original Spectrogram & PSD
log_V = np.log2(V + 1e-11) # Saved to reuse for distribution plotting
ax_orig_spectro.pcolormesh(t_spectro, f_spectro, log_V, shading='nearest', cmap='jet')
ax_orig_spectro.set_title('Original Spectrogram')
ax_orig_spectro.set_ylabel('Frequency (Hz)')

orig_psd = np.mean(V, axis=1)
ax_orig_psd.plot(orig_psd**0.1, f_spectro, color='darkred', lw=2)
ax_orig_psd.set_title('Original PSD')
ax_orig_psd.set_xlabel('Mean Amplitude')
ax_orig_psd.grid(True)

# 2. NMF Components
for i in range(n_components):
    ax_w, ax_h = ax_comps[i]
    ax_w.plot(f_spectro, W[:, i], color='darkorange', lw=2)
    ax_w.set_title(f'Component {i+1} Frequency Profile (W)')
    ax_w.set_xlabel('Frequency (Hz)')
    ax_w.set_ylabel('Amplitude')
    ax_w.grid(True)
    
    ax_h.plot(t_spectro, H[i, :], color='dodgerblue', lw=2)
    ax_h.set_title(f'Component {i+1} Time Activation (H)')
    ax_h.set_ylabel('Activation')
    ax_h.grid(True)

# 3. Reconstructed Spectrogram & PSD
log_V_approx = np.log2(V_approx + 1e-11) # Saved to reuse for distribution plotting
ax_recon_spectro.pcolormesh(t_spectro, f_spectro, log_V_approx, shading='nearest', cmap='jet')
ax_recon_spectro.set_title('Reconstructed Spectrogram (W * H)')
ax_recon_spectro.set_xlabel('Time (s)')
ax_recon_spectro.set_ylabel('Frequency (Hz)')

recon_psd = np.mean(V_approx, axis=1)
ax_recon_psd.plot(recon_psd, f_spectro, color='darkred', lw=2)
ax_recon_psd.set_title('Reconstructed PSD')
ax_recon_psd.set_xlabel('Mean Amplitude')
ax_recon_psd.grid(True)


# --- 4. NEW: Plot Bin Distributions ---
# Flatten the 2D arrays to 1D vectors of pixel/bin values
bins_orig_flat = log_V.flatten()
bins_recon_flat = log_V_approx.flatten()

# Original Spectrogram Bin Distribution
ax_dist_orig.hist(bins_orig_flat, bins=60, color='purple', alpha=0.7, edgecolor='black')
ax_dist_orig.set_title('Original Bin Value Distribution (Log2 Space)')
ax_dist_orig.set_xlabel('Log2(Amplitude)')
ax_dist_orig.set_ylabel('Bin Count')
ax_dist_orig.grid(True)

# Reconstructed Spectrogram Bin Distribution
ax_dist_recon.hist(bins_recon_flat, bins=60, color='teal', alpha=0.7, edgecolor='black')
ax_dist_recon.set_title('Reconstructed Distribution')
ax_dist_recon.set_xlabel('Log2(Amplitude)')
ax_dist_recon.grid(True)

# Clean up layout handling
plt.tight_layout()
plt.show()

In [46]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_multiotsu

# --- 1. Simulate a PSD (Noise floor + 1 weak signal + 1 strong signal) ---
np.random.seed(42)
frequencies = np.linspace(0, 1000, 500)
# Background noise
psd = np.random.exponential(scale=1.0, size=500) 
# Add a weak signal
psd[150:160] += 8.0  
# Add a strong signal
psd[350:360] += 25.0 

# Highly recommended: Apply Multi-Otsu in the Log/Decibel domain 
# This scales the power linearly, making the histogram clusters much tighter
psd_db = 10 * np.log10(psd + 1e-11)

# --- 2. Apply Multi-Otsu ---
# Let's find 2 thresholds to split the PSD into 3 classes: Noise, Weak, and Strong
thresholds = threshold_multiotsu(psd_db, classes=3)

# --- 3. Visualize ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: The PSD with Threshold Lines
ax1.plot(frequencies, psd_db, color='navy', label='PSD (dB)')
ax1.axhline(thresholds[0], color='orange', linestyle='--', lw=2, label=f'Threshold 1: {thresholds[0]:.1f} dB')
ax1.axhline(thresholds[1], color='red', linestyle='--', lw=2, label=f'Threshold 2: {thresholds[1]:.1f} dB')
ax1.set_title('Multi-Otsu Thresholds on PSD')
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Power (dB)')
ax1.legend()
ax1.grid(True)

# Plot 2: The Histogram showing where the lines split the data
ax2.hist(psd_db, bins=40, color='gray', alpha=0.7, edgecolor='black')
ax2.axvline(thresholds[0], color='orange', linestyle='--', lw=2)
ax2.axvline(thresholds[1], color='red', linestyle='--', lw=2)
ax2.set_title('Power Distribution & Threshold Cuts')
ax2.set_xlabel('Power (dB)')
ax2.set_ylabel('Bin Count')
ax2.grid(True)

plt.tight_layout()
plt.show()

## 2. Transform of the data

We perform various transforms on the data to visualize which one emphasize more the horizontal band in $\beta$ regime. We can also swap easily the matrix input from a transform to check its effect on the method efficiency. 

In [ ]:
M = np.log2(spectro + 1e-11) # standard input

# --- 1. Base Pipeline ---
# M_gauss: M filtered through a 5x5 Gaussian kernel
M_gauss = cv2.GaussianBlur(M, (5, 5), sigmaX=0)

# M_grad: Gradient magnitude of original M using 5x5 Sobel
M_grad = np.abs(cv2.Sobel(M, cv2.CV_64F, 1, 1, ksize=5))

# M_grad_gauss: M_grad filtered through a 5x5 Gaussian filter
M_grad_gauss = cv2.GaussianBlur(M_grad, (5, 5), sigmaX=0)

# --- 2. New Pipeline (Gradient of Gauss) ---
# M_gauss_grad: Gradient magnitude computed FROM the blurred image M_gauss
M_gauss_grad = np.abs(cv2.Sobel(M_gauss, cv2.CV_64F, 1, 1, ksize=5))

# M_gauss_grad_gauss: The blurred version of the gradient of gauss
M_gauss_grad_gauss = cv2.GaussianBlur(M_gauss_grad, (5, 5), sigmaX=0)


# --- 3. Visualization (Expanded to 2x3 Grid) ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# --- Row 1: Operations starting from original M ---
axes[0, 0].pcolormesh(M, shading='nearest', cmap='jet')
axes[0, 0].set_title("Original M")

axes[0, 1].pcolormesh(M_grad, shading='nearest', cmap='jet')
axes[0, 1].set_title("M_grad (Gradient of M)")

axes[0, 2].pcolormesh(M_grad_gauss, shading='nearest', cmap='jet')
axes[0, 2].set_title("M_grad_gauss (Smoothed)")

# --- Row 2: Operations starting from blurred M_gauss ---
axes[1, 0].pcolormesh(M_gauss, shading='nearest', cmap='jet')
axes[1, 1].set_title("M_gauss (Blurred)")

axes[1, 1].pcolormesh(M_gauss_grad, shading='nearest', cmap='jet')
axes[1, 1].set_title("M_gauss_grad (Gradient of Gauss)")

axes[1, 2].pcolormesh(M_gauss_grad_gauss, shading='nearest', cmap='jet')
axes[1, 2].set_title("M_gauss_grad_gauss (Smoothed)")


# --- Axis Formatting ---
for ax in axes.ravel():
    ax.invert_yaxis()    # Corrects vertical orientation for spectrogram data
    ax.set_aspect('auto') # Allows the spectrogram time/freq dimensions to scale naturally
    ax.axis('off')       # Hides background grid lines

plt.tight_layout()
plt.show()

## 3. Wavelet decomposition

The discrete wavelet decomposition in 2D can be use to emphasize spoecific patterns in the spectrogram image like horizontal bands. We start to analyise the effect of a 1 level decomposition below.

In [ ]:
coeffs = pywt.dwt2(M, 'db1')
cA, (cH, cV, cD) = coeffs

n_f, n_t = np.shape(cA) # shape of 2D DWT coefficients
t_wav = np.linspace(t_spectro[0], t_spectro[-1], n_t)
f_wav = np.linspace(f_spectro[0], f_spectro[-1], n_f)


fig, axes = plt.subplots(2, 3, sharex = True, constrained_layout = True)
axes[0, 0].plot(t, y)
axes[0, 0].set_title('Simulated EEG signal')
axes[1, 0].pcolormesh(t_spectro, f_spectro, M, shading = 'nearest', cmap = 'jet')
axes[1, 0].set_title('Spectrogram')

axes[0, 1].pcolormesh(t_wav, f_wav, cA, shading = 'nearest', cmap = 'jet')
axes[0, 1].set_title('cA coeff')
axes[1, 1].pcolormesh(t_wav, f_wav, cH, shading = 'nearest', cmap = 'jet')
axes[1, 1].set_title('cH coeff')

axes[0, 2].pcolormesh(t_wav, f_wav, cV, shading = 'nearest', cmap = 'jet')
axes[0, 2].set_title('cV coeff')
axes[1, 2].pcolormesh(t_wav, f_wav, cD, shading = 'nearest', cmap = 'jet')
axes[1, 2].set_title('cD coeff')

plt.show()


To investigate further we switch to a 3 level decomposition.

In [ ]:
# Define the number of decomposition levels you want
levels = 3

# 1. Multi-level 2D Discrete Wavelet Decomposition
# coeffs list structure: [cA_n, (cH_n, cV_n, cD_n), ..., (cH_1, cV_1, cD_1)]
coeffs = pywt.wavedec2(M_gauss, 'db1', level=levels)

# Extract the main approximation coefficient (deepest level)
cA = coeffs[0]

# --- Plotting Setup ---
# We will create a grid where:
# Row 0 handles the original Signal and Spectrogram
# Rows 1 to 'levels' will display cA, cH, cV, and cD for each level
fig, axes = plt.subplots(levels + 1, 4, figsize=(14, 3 * (levels + 1)), constrained_layout=True)

# Turn off any unused axes in the top row to keep it clean
for col in range(2, 4):
    axes[0, col].axis('off')

# 2. Plot Original Signal & Spectrogram (Top Row)
axes[0, 0].plot(t, y)
axes[0, 0].set_title('Simulated EEG signal')

axes[0, 1].pcolormesh(t_spectro, f_spectro, M, shading='nearest', cmap='jet')
axes[0, 1].set_title('Original Spectrogram (M)')

# 3. Dynamically Plot Coefficients for Each Level
for i in range(levels):
    # Map the coefficients correctly from the wavedec2 output list
    # Level 3 is index 1, Level 2 is index 2, Level 1 is index 3
    cH, cV, cD = coeffs[i + 1]
    
    # Determine current level label (e.g., Level 3 down to Level 1)
    current_level = levels - i 
    
    # Find the shape for the current level's coefficient matrix
    n_f, n_t = np.shape(cH)
    t_wav = np.linspace(t_spectro[0], t_spectro[-1], n_t)
    f_wav = np.linspace(f_spectro[0], f_spectro[-1], n_f)
    
    # Row index in our plot grid for this level
    row = i + 1
    
    # Plot cA only for the deepest level grid slot (or keep it aligned per row)
    # Note: wavedec2 only produces ONE final cA at the deepest level (Level 3 here)
    if i == 0:
        n_f_cA, n_t_cA = np.shape(cA)
        t_wav_cA = np.linspace(t_spectro[0], t_spectro[-1], n_t_cA)
        f_wav_cA = np.linspace(f_spectro[0], f_spectro[-1], n_f_cA)
        axes[row, 0].pcolormesh(t_wav_cA, f_wav_cA, cA, shading='nearest', cmap='jet')
        axes[row, 0].set_title(f'cA{current_level} (Approximation)')
    else:
        # Hide the cA column for subsequent rows since it's only calculated once at the bottom
        axes[row, 0].axis('off')
        axes[row, 0].set_title(f'cA{current_level} (N/A)')

    # Plot details: Horizontal, Vertical, Diagonal
    axes[row, 1].pcolormesh(t_wav, f_wav, cH, shading='nearest', cmap='jet')
    axes[row, 1].set_title(f'cH{current_level} (Horizontal Detail)')
    
    axes[row, 2].pcolormesh(t_wav, f_wav, cV, shading='nearest', cmap='jet')
    axes[row, 2].set_title(f'cV{current_level} (Vertical Detail)')
    
    axes[row, 3].pcolormesh(t_wav, f_wav, cD, shading='nearest', cmap='jet')
    axes[row, 3].set_title(f'cD{current_level} (Diagonal Detail)')

plt.show()

## 4. Adaptive thresholding

## Segmentation by PSD baseline

On the PSD, considering a method of detection of the beta bump is already provided, one can fit a baseline to this bump. Now any data higher than the corresponding baseline at a given frequency can be considered part of a burst of activity. Now they should be a part either including the fact that its close neigbour and pixel form an overall blob higher than psd. And/or include a flexibility of the threshold. 

# Correction methods

In [101]:
mask_f = f_spectro >= 20
f_spec = f_spectro[mask_f]
spec = spectro[mask_f, :]

In [86]:
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spec, np.log2(np.sum(spec, axis = 1)))
axes[2].set_title('PSD')
axes[1].sharex(axes[0])

plt.show()

In [100]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Compute the raw PSD (sum of spectrogram power over time for each frequency)
# Reshape to (len(f_spec), 1) to allow broadcasting across the time axis
psd = np.mean(spec, axis=1, keepdims=True)

# Avoid division by zero in case of silent bins
psd_safe = np.where(psd == 0, 1e-11, psd)

# 2. Compute the relative ratio for each bin
# This yields an array of the same shape as 'spec'
relative_ratios = spec / psd_safe

# Flatten the 2D array of ratios to a 1D array for the distribution plot
flat_ratios = relative_ratios.flatten()

# --- Plotting ---
# We will add a 4th subplot to show the distribution
fig, axes = plt.subplots(4, 1, figsize=(8, 10), constrained_layout=True)

# Subplot 1: EEG Signal
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')

# Subplot 2: Spectrogram
axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet')
axes[1].set_title('Spectrogram')
axes[1].sharex(axes[0])

# Subplot 3: PSD
axes[2].plot(f_spec, np.log2(psd.flatten() + 1e-11))
axes[2].set_title('PSD ($\log_2$ scale)')

# Subplot 4: Distribution of Relative Ratios
# We use a log scale for the Y-axis (counts) as ratio distributions are often highly skewed
axes[3].hist(flat_ratios, bins=100, color='purple', edgecolor='black', log=True)
axes[3].set_title('Distribution of Relative Ratios (Spectrogram Bin / PSD)')
axes[3].set_xlabel('Relative Ratio Value')
axes[3].set_ylabel('Frequency Count (Log Scale)')

plt.show()

In [102]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Compute the PSD Reference Line ---
# Option A: Using the average (mean) power per frequency (Recommended for thresholding)
psd_ref = np.median(spec, axis=1, keepdims=True)

# Option B: Using the total sum over time (Uncomment below if this is strictly what you need)
# psd_ref = np.sum(spec, axis=1, keepdims=True)


# --- 2. Create the Mask ---
# This creates a boolean matrix (True where the bin is higher than its corresponding PSD value)
# Using keepdims=True above allows NumPy to automatically broadcast the 1D PSD across all time columns
mask = spec > 2*psd_ref


# --- 3. Get the Thresholded Spectrogram ---
# np.where(condition, value_if_true, value_if_false)
# Bins that don't pass the threshold are set to 0 (you can also use np.nan if preferred)
spec_thresholded = np.where(mask, spec, 0)


# --- 4. Visualization (Optional) ---
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, constrained_layout=True)

# Plot 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet')
axes[0].set_title('Original Spectrogram')
fig.colorbar(im0, ax=axes[0], label='$\log_2(\text{Power})$')

# Plot 2: Thresholded Spectrogram
# We use a safe log transform to avoid log(0) on the thresholded-out bins
spec_thresholded_log = np.log2(np.where(spec_thresholded > 0, spec_thresholded, 1e-11))
im1 = axes[1].pcolormesh(t_spectro, f_spec, spec_thresholded_log, shading='nearest', cmap='jet')
axes[1].set_title('Thresholded Spectrogram (Bins > PSD Reference)')
fig.colorbar(im1, ax=axes[1], label='$\log_2(\text{Power})$')

plt.show()

In [109]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# --- 1. Define the Whittaker-Eilers (Asymmetric Least Squares) Filter ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z


# --- 2. Compute baseline and mask thresholding ---
# Compute the raw PSD (summed across time)
psd_ref_1d = np.median(spec, axis=1)  # shape: (len(f_spec),)

# Compute the Whittaker baseline for the 1D PSD curve
baseline_1d = whittaker_baseline(psd_ref_1d, lmbda=1e5, p=0.01)
baseline_2d = baseline_1d[:, np.newaxis]

# Distribute the baseline evenly across the time bins for 2D comparison
n_time_bins = spec.shape[1]
baseline_per_time_bin = baseline_2d #/ n_time_bins

# Segregate spectrogram bins into higher and lower
mask_higher = spec > baseline_per_time_bin
mask_lower = ~mask_higher

spec_higher = np.where(mask_higher, spec, 0)
spec_lower = np.where(mask_lower, spec, 0)


# --- 3. Compute 1D Projections onto the Frequency Axis ---
proj_higher = np.median(spec_higher, axis=1)
proj_lower = np.median(spec_lower, axis=1)


# --- 4. Plotting (Keeping Spectrograms & Adding Projections) ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), constrained_layout=True)

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet')
axes[0].set_title('Original Spectrogram')
axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(im0, ax=axes[0], label=r'$\log_2(\text{Power})$')

# Panel 2: Thresholded Spectrogram (Only bins > baseline)
spec_thresholded_log = np.log2(np.where(spec_higher > 0, spec_higher, 1e-11))
im1 = axes[1].pcolormesh(t_spectro, f_spec, spec_thresholded_log, shading='nearest', cmap='jet')
axes[1].set_title('Thresholded Spectrogram (Bins > Whittaker Baseline)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].sharex(axes[0])  # Link time axis with original spectrogram
fig.colorbar(im1, ax=axes[1], label=r'$\log_2(\text{Power})$')

# Panel 3: 1D Projections Overlay
axes[2].plot(f_spec, np.log2(psd_ref_1d + 1e-11), 
            label='Total PSD', color='black', linewidth=1.5)
axes[2].plot(f_spec, np.log2(baseline_1d + 1e-11), 
            label='Whittaker Baseline (Noise Floor)', color='red', linestyle='--', linewidth=2)
axes[2].plot(f_spec, np.log2(proj_higher + 1e-11), 
            label='Projection: Bins > Baseline (Signal)', color='forestgreen', alpha=0.8)
axes[2].plot(f_spec, np.log2(proj_lower + 1e-11), 
            label='Projection: Bins < Baseline (Background)', color='royalblue', alpha=0.8)

axes[2].set_title('PSD Projections on Frequency Axis')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel(r'$\log_2(\text{Power})$')
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='best')

plt.show()

In [113]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.ndimage import gaussian_filter

# --- 1. Define the Whittaker-Eilers Filter ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# --- 2. Setup Baseline and Masks ---
psd_ref_1d = np.sum(spec, axis=1)
baseline_1d = whittaker_baseline(psd_ref_1d, lmbda=1e5, p=0.01)
baseline_2d = baseline_1d[:, np.newaxis]

n_time_bins = spec.shape[1]
baseline_per_time_bin = baseline_2d / n_time_bins

# Binary mask of bins above the baseline threshold
mask_higher = spec > 2*baseline_per_time_bin


# --- 3. Peak Correction with Constraints ---
# A) Peak Compression (Preserves local maxima and shapes)
# We pull the peak height down towards the baseline by a scale factor (e.g., 85% compression)
compression_factor = 0.05  # 1.0 = unchanged, 0.0 = completely flattened to baseline
excess_power = spec - baseline_per_time_bin

# Reconstruct power: keep baseline and scale down only the peak excursions
spec_corrected_raw = np.where(
    mask_higher, 
    baseline_per_time_bin + (excess_power * compression_factor), 
    spec
)

# B) Smooth Transition Boundary (Gaussian blending at the mask edges)
# We smooth the binary mask to create a feathering gradient weight (0 to 1)
smooth_weight = gaussian_filter(mask_higher.astype(float), sigma=1.5)

# Blend the original and corrected spectrograms using this boundary weight
spec_reconstructed = (1 - smooth_weight) * spec + smooth_weight * spec_corrected_raw


# --- 4. Calculate Reconstructed PSD ---
psd_reconstructed = np.sum(spec_reconstructed, axis=1)


# --- 5. Visualization ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), constrained_layout=True)

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet')
axes[0].set_title('Original Spectrogram')
axes[0].set_ylabel('Frequency (Hz)')
fig.colorbar(im0, ax=axes[0], label=r'$\log_2(\text{Power})$')

# Panel 2: Reconstructed/Corrected Spectrogram (Smooth boundaries, local maxima intact)
im1 = axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec_reconstructed + 1e-11), shading='nearest', cmap='jet')
axes[1].set_title('Reconstructed Spectrogram (Peaks Compressed & Blended)')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].sharex(axes[0])
fig.colorbar(im1, ax=axes[1], label=r'$\log_2(\text{Power})$')

# Panel 3: Comparison of PSDs and Baseline
axes[2].plot(f_spec, np.log2(psd_ref_1d + 1e-11), 
            label='Original PSD', color='black', alpha=0.5, linewidth=1.5)
axes[2].plot(f_spec, np.log2(baseline_1d + 1e-11), 
            label='Target Whittaker Baseline', color='red', linestyle='--', linewidth=2)
axes[2].plot(f_spec, np.log2(psd_reconstructed + 1e-11), 
            label='Reconstructed PSD (Closest to Fit)', color='forestgreen', linewidth=2)

axes[2].set_title('PSD Correction Convergence Analysis')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel(r'$\log_2(\text{Power})$')
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc='best')

plt.show()

In [120]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.optimize import minimize  # Use SciPy's built-in optimizer

# --- 1. Whittaker-Eilers Baseline ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# --- 2. Baseline Extraction ---
psd_ref_1d = np.sum(spec, axis=1)
baseline_1d = whittaker_baseline(psd_ref_1d, lmbda=1e5, p=0.01)
baseline_2d = baseline_1d[:, np.newaxis]

n_freqs, n_times = spec.shape
baseline_per_time_bin = baseline_2d / n_times

# --- 3. Optimization Setup using SciPy SLSQP (Vectorized) ---
spec_reconstructed = np.zeros_like(spec)
gamma_val = 0.1  # Regularization parameter

print("Optimizing spectrogram slices using vectorized SLSQP...")
for t in range(n_times):
    x_t = spec[:, t]
    b_t = baseline_per_time_bin
    
    # Pre-calculate target slope signs for the entire slice (shape preservation)
    # +1 if the original next bin is higher/equal, -1 if it is lower
    original_diffs = np.diff(x_t)
    signs_t = np.where(original_diffs >= 0, 1.0, -1.0)
    
    # 1. Vectorized Objective Function
    def objective(hat_x):
        term1 = 0.5 * np.sum((hat_x - b_t) ** 2)
        term2 = 0.5 * gamma_val * np.sum((hat_x - x_t) ** 2)
        return term1 + term2
    
    # 2. Vectorized Slope Constraint
    # Instead of N-1 separate functions, we return a single array of length N-1.
    # SLSQP natively understands that every element of the returned array must be >= 0.
    def slope_constraint(hat_x):
        return signs_t * (hat_x[1:] - hat_x[:-1])
    
    constraints = [{'type': 'ineq', 'fun': slope_constraint}]
    
    # 3. Bounds: Keeping background locked and peaks above baseline
    bounds = []
    for i in range(n_freqs):
        if x_t[i] <= b_t[i]:
            bounds.append((x_t[i], x_t[i]))  # Locked
        else:
            bounds.append((b_t[i], None))     # >= baseline
            
    # Solve the optimization problem
    res = minimize(
        objective, 
        x0=x_t.copy(),  # Warm start with original spectrogram slice
        method='SLSQP', 
        bounds=bounds, 
        constraints=constraints,
        options={'maxiter': 150, 'ftol': 1e-6}
    )
    
    spec_reconstructed[:, t] = res.x

# --- 4. Calculate Final Reconstructed PSD ---
psd_reconstructed = np.sum(spec_reconstructed, axis=1)

# --- 5. Plotting ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), constrained_layout=True)

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet')
axes[0].set_title('Original Spectrogram')
fig.colorbar(im0, ax=axes[0], label=r'$\log_2(\text{Power})$')

# Panel 2: Vectorized SciPy SLSQP Optimized Spectrogram
im1 = axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec_reconstructed + 1e-11), shading='nearest', cmap='jet')
axes[1].set_title('Optimized Spectrogram (Vectorized SLSQP Shape-Constrained)')
axes[1].sharex(axes[0])
fig.colorbar(im1, ax=axes[1], label=r'$\log_2(\text{Power})$')

# Panel 3: PSD Comparison
axes[2].plot(f_spec, np.log2(psd_ref_1d + 1e-11), label='Original PSD', color='black', alpha=0.5)
axes[2].plot(f_spec, np.log2(baseline_1d + 1e-11), label='Target Whittaker Baseline', color='red', linestyle='--')
axes[2].plot(f_spec, np.log2(psd_reconstructed + 1e-11), label='SciPy Reconstructed PSD', color='forestgreen', linewidth=2)
axes[2].set_title('SciPy Optimized PSD Reconstruction')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel(r'$\log_2(\text{Power})$')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.show()

Optimizing spectrogram slices using vectorized SLSQP...


In [122]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# --- 1. Whittaker-Eilers Baseline ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# --- 2. Baseline Extraction ---
psd_ref_1d = np.sum(spec, axis=1)
baseline_1d = whittaker_baseline(psd_ref_1d, lmbda=1e5, p=0.01)

n_freqs, n_times = spec.shape
# Baseline value allocated to each individual time bin
b_f = (baseline_1d / n_times)[:, np.newaxis]  # shape: (n_freqs, 1)

# --- 3. Analytical Monotonic Transport Map ---
# Create masks for bins below and above baseline
mask_below = spec <= b_f
mask_above = ~mask_below

# Compute sums required for the analytical solution
sum_below = np.sum(np.where(mask_below, spec, 0), axis=1, keepdims=True)
n_below = np.sum(mask_below, axis=1, keepdims=True)
total_excess = np.sum(np.where(mask_above, spec - b_f, 0), axis=1, keepdims=True)

# Calculate the scaling factor k_f for each frequency (handling division by zero)
numerator = (n_below * b_f) - sum_below
k_f = np.where(total_excess > 0, numerator / total_excess, 1.0)

# Apply the monotonic transformation
spec_reconstructed = np.where(
    mask_above,
    b_f + k_f * (spec - b_f),
    spec
)

# --- 4. Calculate Final Reconstructed PSD ---
psd_reconstructed = np.sum(spec_reconstructed, axis=1)

# --- 5. Plotting ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), constrained_layout=True)

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -4)
axes[0].set_title('Original Spectrogram')
fig.colorbar(im0, ax=axes[0], label=r'$\log_2(\text{Power})$')

# Panel 2: Reconstructed Spectrogram (Perfect Rank & Order Preserved)
im1 = axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec_reconstructed + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -4)
axes[1].set_title('Reconstructed Spectrogram (Monotonic Cumulative Transport)')
axes[1].sharex(axes[0])
fig.colorbar(im1, ax=axes[1], label=r'$\log_2(\text{Power})$')

# Panel 3: PSD Comparison (Green line will sit perfectly on top of the Red dashed line)
axes[2].plot(f_spec, np.log2(psd_ref_1d + 1e-11), label='Original PSD', color='black', alpha=0.4)
axes[2].plot(f_spec, np.log2(baseline_1d + 1e-11), label='Target Whittaker Baseline', color='red', linestyle='--', linewidth=3)
axes[2].plot(f_spec, np.log2(psd_reconstructed + 1e-11), label='Transport Reconstructed PSD', color='forestgreen', linestyle=':', linewidth=2)
axes[2].set_title('PSD Correction Convergence (Zero Error Fit)')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel(r'$\log_2(\text{Power})$')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.show()

In [124]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# --- 1. Whittaker-Eilers Baseline ---
def whittaker_baseline(y, lmbda=1e5, p=0.01, n_iter=10):
    n = len(y)
    D = diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(n-2, n), format='csc')
    w = np.ones(n)
    for _ in range(n_iter):
        W = diags([w], [0], shape=(n, n), format='csc')
        A = W + lmbda * D.T @ D
        z = spsolve(A, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# --- 2. Baseline Extraction ---
psd_ref_1d = np.sum(spec, axis=1)
baseline_1d = whittaker_baseline(psd_ref_1d, lmbda=1e5, p=0.01)

n_freqs, n_times = spec.shape
b_f = (baseline_1d / n_times)[:, np.newaxis]  # shape: (n_freqs, 1)

# --- 3. Bounded Cumulative Transport Map ---
mask_above = spec > b_f
mask_below = ~mask_above

# Extract original boundaries for peak bins at each frequency
# Replacing non-peak values with NaNs to cleanly find min/max of peaks per row
peaks_only = np.where(mask_above, spec, np.nan)
u_max = np.nanmax(peaks_only, axis=1, keepdims=True)
u_min = np.nanmin(peaks_only, axis=1, keepdims=True)

# Replace NaNs with a safe fallback in case a frequency band has no peaks
u_max = np.nan_to_num(u_max, nan=1e-11)
u_min = np.nan_to_num(u_min, nan=1e-11)

# Define how close we want the corrected bounds to be to the uncorrected bounds
# alpha = 1.0 means the corrected max/min matches the uncorrected max/min exactly
# alpha = 0.5 means they are compressed halfway down toward the baseline
alpha_min = 0.9  # Keeping corrected minimum close to uncorrected minimum
alpha_max = 0.7  # Keeping corrected maximum close to uncorrected maximum

# Set target boundaries for corrected values
c_max_target = b_f + alpha_max * (u_max - b_f)
c_min_target = b_f + alpha_min * (u_min - b_f)

# Prevent division by zero if u_max == u_min
range_u = np.where((u_max - u_min) == 0, 1e-11, u_max - u_min)

# Standardized scaling variable (maps original peaks to a 0-1 range)
S = np.where(mask_above, (spec - u_min) / range_u, 0.0)

# Calculate the sums needed for the exact baseline constraint
# Sum of untouched background bins
sum_below = np.sum(np.where(mask_below, spec, 0), axis=1, keepdims=True)
# Count of bins above baseline
n_above = np.sum(mask_above, axis=1, keepdims=True)
# Sum of the standardized scaling variables for peak bins
sum_S = np.sum(S, axis=1, keepdims=True)

# Calculate the precise adjustments to satisfy: Sum(Corrected_Peaks) + Sum(Background) = Target_Baseline
# We scale the range (c_max - c_min) so that cumulative target sum is achieved perfectly
target_peak_sum = baseline_1d[:, np.newaxis] - sum_below
denominator = n_above * c_min_target + sum_S * (c_max_target - c_min_target)
k_scale = np.where(denominator > 0, target_peak_sum / denominator, 1.0)

# Apply dynamic transport limits
c_min_final = c_min_target * k_scale
c_max_final = c_max_target * k_scale

# Reconstruct using the exact transport map
spec_reconstructed = np.where(
    mask_above,
    c_min_final + S * (c_max_final - c_min_final),
    spec
)

# --- 4. Calculate Reconstructed PSD ---
psd_reconstructed = np.sum(spec_reconstructed, axis=1)

# --- 5. Plotting ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), constrained_layout=True)

# Panel 1: Original Spectrogram
im0 = axes[0].pcolormesh(t_spectro, f_spec, np.log2(spec + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -4)
axes[0].set_title('Original Spectrogram')
fig.colorbar(im0, ax=axes[0], label=r'$\log_2(\text{Power})$')

# Panel 2: Reconstructed Spectrogram (Boundaries preserved)
im1 = axes[1].pcolormesh(t_spectro, f_spec, np.log2(spec_reconstructed + 1e-11), shading='nearest', cmap='jet', vmin = -20, vmax = -4)
axes[1].set_title('Reconstructed Spectrogram (Bounded Cumulative Transport)')
axes[1].sharex(axes[0])
fig.colorbar(im1, ax=axes[1], label=r'$\log_2(\text{Power})$')

# Panel 3: PSD Comparison
axes[2].plot(f_spec, np.log2(psd_ref_1d + 1e-11), label='Original PSD', color='black', alpha=0.4)
axes[2].plot(f_spec, np.log2(baseline_1d + 1e-11), label='Target Whittaker Baseline', color='red', linestyle='--', linewidth=3)
axes[2].plot(f_spec, np.log2(psd_reconstructed + 1e-11), label='Transport Reconstructed PSD', color='forestgreen', linestyle=':', linewidth=2)
axes[2].set_title('PSD Correction Convergence (Zero Error Fit)')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel(r'$\log_2(\text{Power})$')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.show()